In [ ]:
curl -sSL https://install.python-poetry.org | POETRY_HOME=/data/bma/envs/poetry python3 -
cd /data/bma/nicheflow && /data/bma/envs/poetry/bin/poetry install
source /data/bma/envs/poetry/venv/bin/activate

In [ ]:
# 基础依赖
pip install torch==2.5.1 scanpy==1.11.4 numpy==2.3.3 tqdm==4.67.1 -i https://pypi.tuna.tsinghua.edu.cn/simple --no-cache-dir
pip install torch-geometric==2.6.1 scipy==1.16.2 matplotlib==3.10.6 flax==0.11.2 diffrax==0.7.0 jaxtyping==0.3.2 -i https://pypi.tuna.tsinghua.edu.cn/simple --no-cache-dir
pip install lightning==2.5.5 torchmetrics==1.8.2 hydra-core==1.3.2 hydra-submitit-launcher==1.2.0 wandb==0.22.0 torchcfm==1.0.7 -i https://pypi.tuna.tsinghua.edu.cn/simple --no-cache-dir
pip install torchdyn==1.0.6 tabulate==0.9.0 omegaconf ruff>=0.13.1 mypy>=1.18.2 ipykernel>=6.30.1 -i https://pypi.tuna.tsinghua.edu.cn/simple --no-cache-dir


# 安装 moscot（需要从 git 安装）
pip install git+https://github.com/theislab/moscot.git

In [ ]:
wget -p /workspace/nicheflow/data https://doi.org/10.6084/m9.figshare.30426610/files/data

### train

训练分为两类不同的任务，它们的 epoch/step 设置机制各不相同：

用于评估的分类器模型训练 (Classifier)：
分类器任务（位于 experiment/classifier/... 下，包含了对于 abd, mba, med 数据集的任务）是按 Epoch 进行训练的。
在这些配置中硬编码了：
max_epochs: 500
这意味着如果是跑评估用的分类器模型，一次完整的训练会跑 500 个 epochs。

核心 Flow Matching 模型训练（NicheFlow / RPCFlow / SPFlow）：
所有的流匹配主体模型（如你之前运行的 experiment=nicheflow/glvfm/med 等），使用了特殊的 trainer: iterative_gpu 配置。如我们在上一个问题中看到的，此类模型并不按 epoch 结束。它们在 iterative_gpu.yaml 中被显式地指定了采用迭代步数控制：
max_steps: 100_000
这意味着模型会在 PyTorch Lightning 的 Infinite DataLoader 设定下一直跑，直到全局优化步数达到 100,000 步 (steps) 为止，这期间 epoch 的统计数字实际上是被忽略或失去限制意义的。

In [ ]:
# python nicheflow/train.py experiment=nicheflow/glvfm/med
cd /workspace/nicheflow && python -m nicheflow.train experiment=nicheflow/glvfm/med
cd /workspace/nicheflow && python -m nicheflow.train experiment=nicheflow/gvfm/med
cd /workspace/nicheflow && python -m nicheflow.train experiment=nicheflow/cfm/med

In [ ]:
# tmux
tmux ls
# gpu
watch -n 1 nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv
nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total --format=csv
# 
tmux a -t nicheflow_gvfm_med
tmux kill-session -t rpcflow_glvfm_med

并行运行了6组实验，GPU utilization（利用率）通常不会同时达到最大值，原因可能有以下几点：
数据加载瓶颈：如果数据预处理或加载速度跟不上，GPU 会等待数据，导致利用率不高。可以尝试增大 DataLoader 的 num_workers 参数。
单卡单进程的 PyTorch Lightning 默认设置：每个实验只用一个进程和一个 GPU，且大部分时间在前向/反向传播，I/O、同步等阶段 GPU 会空闲。
显存占用≠计算利用率：即使显存被占满，只有在进行矩阵运算时 GPU utilization 才会上升，其他时间可能很低。
CPU 性能瓶颈：如果 CPU 性能不足或线程数太少，也会拖慢数据准备，导致 GPU 等待。
任务本身计算量不大：如果模型较小或 batch size 较小，单次迭代计算量有限，GPU utilization 难以拉满。
进程调度与资源争抢：多个进程同时争用 PCIe、内存、磁盘等资源，可能导致每个进程都不能充分利用 GPU。

In [ ]:
tmux new-session -s nicheflow_glvfm_med
tmux new-session -s nicheflow_gvfm_med
tmux new-session -s nicheflow_cfm_med

conda activate /data/bma/envs/nicheflow && export TMPDIR=/data/bma/pkgs/tmp && cd /data/bma/nicheflow

CUDA_VISIBLE_DEVICES=0 python -m nicheflow.train experiment=nicheflow/glvfm/med
#  outputs/2026-03-02/20-46-28/wandb/offline-run-20260302_204632-2u0nnruh/logs
CUDA_VISIBLE_DEVICES=1 python -m nicheflow.train experiment=nicheflow/gvfm/med
CUDA_VISIBLE_DEVICES=2 python -m nicheflow.train experiment=nicheflow/cfm/med



In [ ]:
tmux new-session -s rpcflow_glvfm_med
tmux new-session -s rpcflow_gvfm_med
tmux new-session -s rpcflow_cfm_med

conda activate /data/bma/envs/nicheflow && export TMPDIR=/data/bma/pkgs/tmp && cd /data/bma/nicheflow

CUDA_VISIBLE_DEVICES=4 python -m nicheflow.train experiment=rpcflow/glvfm/med
CUDA_VISIBLE_DEVICES=5 python -m nicheflow.train experiment=rpcflow/gvfm/med
CUDA_VISIBLE_DEVICES=6 python -m nicheflow.train experiment=rpcflow/cfm/med

In [ ]:
nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total --format=csv
watch -n 1 nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv


In [ ]:
Epoch 0:   1%| | 1000/100000 [12:39<20:53:23,  1.32it/s, v_num=iu43, train/loss=0.436, train/loss_x=0.350, train/loss_pos=Your vector field does not have `nn.Parameters` to optimize.                                          | 0/8 [00:00<?, ?it/s]
/workspace/envs/nicheflow/lib/python3.11/site-packages/torchdyn/numerics/odeint.py:83: UserWarning: Setting tolerances has no effect on fixed-step methods
  warn("Setting tolerances has no effect on fixed-step methods")
/workspace/envs/nicheflow/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 400. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
                                                                                                                          Your vector field does not have `nn.Parameters` to optimize.                                  | 1/8 [00:00<00:06,  1.16it/s]
                                                                                                                          Your vector field does not have `nn.Parameters` to optimize.                                  | 2/8 [00:01<00:04,  1.20it/s]
                                                                                                                          Your vector field does not have `nn.Parameters` to optimize.                                  | 3/8 [00:02<00:04,  1.22it/s]
                                                                                                                          Your vector field does not have `nn.Parameters` to optimize.██▌                               | 4/8 [00:03<00:03,  1.23it/s]
                                                                                                                          Your vector field does not have `nn.Parameters` to optimize.██████████▍                       | 5/8 [00:05<00:03,  0.96it/s]
                                                                                                                          Your vector field does not have `nn.Parameters` to optimize.██████████████████▎               | 6/8 [00:07<00:02,  0.84it/s]
                                                                                                                          Your vector field does not have `nn.Parameters` to optimize.██████████████████████████▏       | 7/8 [00:09<00:01,  0.77it/s]

这段输出中并没有真正的报错（Error），而是包含了一些警告（Warning）和提示信息。

在 Python 和深度学习框架中，遇到报错（Exception/Error，如 RuntimeError, TypeError）会直接中断并杀死进程。但遇到警告（Warning）时，框架只是为了提醒你某些设置可能不是最优的或者存在歧义，程序默认会打印出这些提示然后继续运行。

这也是为什么你的训练没有停止并继续出现进度条的原因。

下面为你逐一分析这三段特殊的输出分别是什么意思：

1. ODE求解器步长提示
/.../torchdyn/numerics/odeint.py:83: UserWarning: Setting tolerances has no effect on fixed-step methods

原因：这是 torchdyn（常微分方程求解库）发出的警告。在流匹配（Flow Matching）的采样或推断阶段，需要求解 ODE。这里提示你：当前代码配置使用的是固定步长（Fixed-step）的求解算法（比如 Euler 法），对于固定步长算法，即使代码里设置了误差容忍度（tolerances，如 atol / rtol），也是不起作用的。
影响：完全无害。它仅仅是个提醒，代码依旧按你预设的固定步长执行。
2. PyTorch Lightning 的 Batch Size 识别警告
Trying to infer the batch_size from an ambiguous collection. The batch size we found is 400. To avoid any miscalculations, use self.log(..., batch_size=batch_size).

原因：框架（PyTorch Lightning）在记录和计算 Loss（损失值）等指标时，需要除以 batch_size 求平均。由于我们项目处理的是点云/图数据（用到了 torch-geometric），DataLoader 吐出来的数据格式比较复杂（通常是一个图或者字典嵌套），Lightning 自动推断出的 batch_size 是 400（即 400 个细胞/微环境），但它自己不太确信，所以发出了警告。
影响：正常情况下无害（它猜的400通常是对的）。如果想消除此警告，要在对应代码的 self.log 中显式加上 batch_size=400。
3. 向量场优化提示及子进度条
Your vector field does not have nn.Parameters to optimize.  | 1/8 [00:00<00:06, 1.16it/s]

原因：这也不是报错。它看起来是在模型训练达到第 1000 步（1000/100000）时，触发了一个验证/评估阶段（Validation/Evaluation）或生成图像的 Callback回掉函数。
在评估采样的时候，模型仅仅用来前向推导生成流的轨迹，这时候是不需要/也没有参数需要被优化的（梯度被切断）。所以底层的 ODE 计算组件输出了这句话，并附带了一个跑 8 步的内部进度条（1/8 到 7/8 等）。
影响：这是代码预期内的正常行为。
总结
你的训练一切正常，没有任何致命错误产生。程序只是在第 1000 步时进行了一次常规的流匹配轨迹采样/验证，打印了几个底层依赖库的警告提示，后续它会继续按部就班地完成那 100000 步训练步的。你可以放心地让它接着跑。

### eval

In [ ]:
python python /workspace/nicheflow/nicheflow/eval.py experiment=... ckpt_path=...

## 主要结果
```bash
cd /data/bma/nicheflow && python nicheflow/eval_state_dict_ckpts.py
python -m nicheflow.eval_state_dict_ckpts
```

## $K$ 区域消融实验
```bash
python nicheflow/eval_state_dict_ckpts_kregion_ablations.py
python -m nicheflow.eval_state_dict_ckpts_kregion_ablations
```

## $\lambda$ OT 消融实验
```bash
python nicheflow/eval_state_dict_ckpts_ot_ablations.py
python -m nicheflow.eval_state_dict_ckpts_ot_ablations
```

## 我们的预生成结果

- **主要结果**：下载 `main_results.zip` 并将其解压到 `outputs/eval_main/` 目录中。
- **$K$ 区域消融实验**：下载 `kregion_ablations_results.zip` 并将其解压到 `outputs/eval_kregion_ablations/` 目录中。
- **$\lambda$ OT 消融实验**：下载 `ot_ablation_results.zip` 并将其解压到 `outputs/eval_ot_ablations/` 目录中。

文件放置到位后，使用 [`print_eval_results`](../notebooks/print_eval_results.ipynb) notebook 来复现论文中展示的表格。

# NOTES

整个项目采用了 **PyTorch Lightning + Hydra** 的现代化标准深度学习框架进行构建。以下是该项目代码框架的详细介绍：

### 1. 代码整体执行 Pipeline (工作流)
项目的整个生命周期（从配置读取到模型评估）如下：
1. **统一配置读取：** 通过终端执行 `python nicheflow/train.py experiment=...`。此时，**Hydra** 会解析 `configs/` 文件夹下游的一系列 `.yaml` 配置文件组合（包含优化器、数据路径、模型定义等），动态注入所需参数。
2. **模块实例化：** `train.py` 作为程序入口点，会并行初始化三核组件：
   - **`LightningDataModule`** (比如 `microenv_datamodule.py`)。
   - **`LightningModule`** 即任务包装器 (比如 `tasks/flow_matching.py`)。
   - **`Trainer`** 负责将两者组合并驱动运行。
3. **数据预处理与加载：** 在 `LightningDataModule` 进行准备时，读取经预处理的 H5AD 数据集。数据加载器 `datasets/st_dataset_base.py` 会负责处理复杂的匹配（如通过 `torchcfm` 的 `OTPlanSampler` 使用最优传输计算匹配不同时间节点的细胞微环境配对）。
4. **流匹配与反向传播：** 训练阶段，微环境的点云会作为输入传入模型主干（如 `pc_transformer.py`）。通过预设的 `GLVFMLoss` 或 `GVFMLoss`（`losses.py`）计算连续流匹配的梯度，通过反向传播更新模型。
5. **采样与评估：** 评估阶段（`eval.py`），模型使用 `torchdyn.core.NeuralODE` 求解常微分方程提取生成流，获得目标时间节点的模拟点云，并可能经由预训练好的分类器（`ct_classification.py`）评估生成效果。

---

### 2. 核心文件夹及类文件作用

核心代码主要位于 nicheflow 目录下（注意第一个是根目录，第二个是实际上真正存放 Python Module 的包）：

#### 📍 训练与评测入口
* **`train.py`**：负责读取 Hydra 参数并调用 PyTorch Lightning 的 Trainer 执行 `fit()` 和 `test()` 函数完成训练、保存和测试逻辑。
* **`eval.py` 及相关衍生脚本** (`eval_state_dict_ckpts.py` 等)：使用独立检查点（`.ckpt`）进行生成结果测试及消融实验 (Ablations)。

#### 📍 `nicheflow/tasks/`（任务调度器）
这层由 PyTorch Lightning（`LightningModule`）构成，包装了模型、验证逻辑并定义了 `training_step()` 与 `validation_step()`：
* **`flow_matching.py`**：训练 NicheFlow/RPCFlow/SPFlow 所对应的流匹配任务，控制梯度与核心训练循环。
* **`ct_classification.py`**：训练用于对最终生成的微环境图谱进行度量的下游 Cell Type 评估分类器。

#### 📍 `nicheflow/models/`（模型架构层）
这里存放了核心的底层 `torch.nn.Module` 数学和架构结构：
* **`backbones/pc_transformer.py`**：点云 Transformer 主干网络（模型基于序列或图对微环境点云提取特征），是 NicheFlow / RPCFlow 的核心。
* **`backbones/spmlp.py`**：单点多层感知机（SinglePointMLP），是基线模型 SPFlow（Single Point Flow）的核心。
* **`flows.py`**：定义了构建微分方程生成轨迹的流模型体系架构（`PointCloudFlow` 和 `SinglePointFlow`）。
* **`losses.py`**：不同变分目标的损失函数定义：包含 `CFMLoss`（常规流）、`GVFMLoss`、`GLVFMLoss`（论文核心指标）。
* **`ct_classifier.py`**：细胞类型分类器的网络结构定义。

#### 📍 `nicheflow/datamodules/` 与 `nicheflow/datasets/`（数据层）
使用了 PyTorch Dataset 与 Lightning Datamodule 结合的架构：
* **`microenv_datamodule.py` 及 `microenv_dataset.py`**：用于处理真正的空间点云（微环境）任务格式。
* **`sp_and_rpc_datamodule.py` 及 `rpc_dataset.py`**：用于处理点处理（Single Point）和随机切分点云（Randomized Point Cloud）基线实验的数据。
* **`st_dataset_base.py`**：时序和空间数据特征处理基类，利用了最优传输（Optimal Transport - OT）来生成源域到目标域的 Pair 配对轨迹。
* **`h5ad_ct_dataset.py`**：专门用来训练 Cell Type 分类器的数据加载器，操作 AnnData (`.h5ad`) 文件。

#### 📍 其他支线文件夹
* **`nicheflow/preprocessing/`**：`h5ad_preprocessor.py` 等执行数据清洗，QC 质控并对空间转录图表做标准化抽象。
* **`nicheflow/transforms/`**：定义对 DataLoader 产出时的独立增广算子（如对坐标数据施加自定义 `one_hot_encode_slice.py` 变换）。
* **`nicheflow/utils/`**：辅助性操作，例如固定种子（`seed.py`）、日志辅助仪（`log.py`）、绘图函数（`plots.py`）。

---

### 3. 组件间的关联链路机制（以 NicheFlow 训练为例）

要快速理解整体关联，你可以把整个项目当作一台装配线来看：

1. **装配站初始化 (Hydra configs -> `train.py`):** 
   当你下达带有参数文件的执行命令（例如 `experiment=nicheflow/glvfm/med`）时，`train.py` 第一步就是把配置文件的需求转化为具体的类实例化。它分别呼叫 `microenv_datamodule.py` 生成传送带，和 `flow_matching.py`（任务）作为机床。
2. **建立基础模型结构模型 (`flow_matching.py` -> `models/`)**: 
   `flow_matching.py` 在被初始化时，它会向 `models/` 拿来 `flows.py` 中的组合包装，并且 `flows.py` 内部又加载了来自 `backbones/pc_transformer.py` 作为深度学习可调参数层网络架构，并将 `losses.py` 中的 `GLVFMLoss` 直接接入当作指导其方向的指标函数。
3. **数据流入与计算 (`microenv_datamodule.py` -> `dataset` -> Task):** 
   数据在 `datasets/microenv_dataset.py` 被包装成 Batch 并使用最优传输（OT）算出哪组微环境 $t_0$ 需要向哪个 $t_1$ 配对（生成配对点云）。随后，Batches 会自动被输入到 `tasks/flow_matching.py` 中的 `training_step()` 里。
4. **最终生成输出 (`training_step` -> ODE Solver):**
   模型将处理好的配对交给 `models.PointCloudFlow` 接受推导并给出流向量并计算 Loss 回传。在验证 / 预测输出阶段，网络实际上通过 `torchdyn.core.NeuralODE`（神经微分方程求解器）使用这个 Transformer 逐帧步进（积分）还原出时间段的微环境预测图像图谱，并交由评价组（Classifier）打分。

In [2]:
import pickle; data = pickle.load(open('/data/bma/nicheflow/data/embryonic_data.pkl', 'rb')); print(type(data));

ModuleNotFoundError: No module named 'torch'

- 多GPU：增加devices

---

```YAML
defaults:
  - default

accelerator: gpu
devices: [1, 3]     # 手动指定使用的具体的 GPU ID
# devices: "auto"   # 或-1。它会吃掉它能看见的所有可用卡

strategy: ddp
use_distributed_sampler: True
```
---

- 多CPU：增加num_workers # 设置为你CPU逻辑核心数的 1/4 到 1/2